# Notebook 03: Embedding Generation

Generates all embedding tensors for both ESM-2 and AbLang2 and saves them to Drive.

All cells are idempotent: if a tensor already exists with the correct shape on Drive, the cell prints its shape and skips regeneration.

**Tensors produced:**

| File | Shape | Description |
|---|---|---|
| `esm2_abagym.pt` | (5318, 2560) | ESM-2 mutant sequences, AbAgym |
| `esm2_abagym_wildtype.pt` | (5, 2560) | ESM-2 wildtype sequences, one per antibody |
| `esm2_sabdab.pt` | (491, 2560) | ESM-2 sequences, SAbDab |
| `esm2_abagym_residue_mutsite.pt` | (5318, 1280) | ESM-2 token at mutation site, mutant sequence |
| `esm2_abagym_residue_wtsite.pt` | (5318, 1280) | ESM-2 token at mutation site, wildtype sequence |
| `esm2_abagym_delta.pt` | (5318, 2560) | Sequence-level delta: mutant - wildtype |
| `esm2_abagym_residue_delta.pt` | (5318, 1280) | Residue-level delta: mutsite - wtsite |
| `ablang2_abagym.pt` | (5318, 960) | AbLang2 mutant sequences, AbAgym |
| `ablang2_abagym_wildtype.pt` | (5, 960) | AbLang2 wildtype sequences |
| `ablang2_sabdab.pt` | (491, 960) | AbLang2 sequences, SAbDab |
| `ablang2_abagym_residue_mutsite.pt` | (5318, 480) | AbLang2 token at mutation site, mutant |
| `ablang2_abagym_residue_wtsite.pt` | (5318, 480) | AbLang2 token at mutation site, wildtype |
| `ablang2_abagym_delta.pt` | (5318, 960) | Sequence-level delta: mutant - wildtype |
| `ablang2_abagym_residue_delta.pt` | (5318, 480) | Residue-level delta: mutsite - wtsite |

All tensors are saved to `EMBEDDING_DIR` (Google Drive Desktop mount, auto-resolved by `src/config.py`).

In [1]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    # Local: notebook is in notebooks/, repo root is one level up
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")
print("Ready.")

Environment: local
Repo: /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project
Ready.


In [2]:
# src/config.py resolves DRIVE_ROOT in priority order:
#   1. Colab: /content/drive/MyDrive/DL_Final_Project/Antibody_Project
#   2. Local Mac with Google Drive Desktop mounted (email-agnostic glob)
#   3. Fallback: outputs/ at repo root (gitignored)
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:    {DRIVE_ROOT}")
print(f"Embedding dir: {EMBEDDING_DIR}")
print("Paths set.")

Drive root:    /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project
Embedding dir: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/embeddings
Paths set.


In [3]:
if IN_COLAB:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb'], check=True)
else:
    print("Local run -- installation skipped.")

Local run -- installation skipped.


In [4]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
else:
    print("Local run -- skipped.")

Local run -- skipped.


In [5]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    import subprocess
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

Autoreload enabled.


In [6]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS (Metal Performance Shaders) -- Apple Silicon unified memory")
else:
    print("CPU -- embedding generation will be slow")

Device: mps
Apple MPS (Metal Performance Shaders) -- Apple Silicon unified memory


In [7]:
# wandb not needed for embedding generation.
print("Skipped.")

Skipped.


## Imports

Imports `src/` modules only. `ABAGYM_DATASETS` is the ordered list of 5 antibody names used to fix the wildtype tensor row order (required by `compute_delta_sequence`). `save_index` / `load_index` serialize `{int: str}` mappings as JSON for the wildtype and SAbDab indexes.

In [8]:
import torch
from src.config import DEVICE, EMBEDDING_DIR, DATA_DIR, ABAGYM_DATASETS
from src.utils import save_index, load_index
from src.data.abagym import load_abagym_antibody, load_abagym_sequences, get_all_mutation_site_indices
from src.data.sabdab import load_sabdab
from src.embeddings import esm2, ablang2
from src.embeddings.delta import compute_delta_sequence, compute_delta_residue

print("Imports OK.")

Imports OK.


## Load Data

Loads the three CSVs from `data/` and builds the arrays needed by the embedding functions.

Key outputs:
- `mutant_sequences`: 5318 `(heavy, light)` tuples from `abagym_antibody.csv` -- one per mutation, each with exactly one AA substitution applied
- `wt_expanded_sequences`: same 5318 rows but wildtype sequences (used for wtsite residue extraction -- same site indices, unmodified sequence)
- `wildtype_sequences`: 5 tuples, one per antibody, ordered by `ABAGYM_DATASETS` -- row order must match the wildtype tensor saved to disk so that `compute_delta_sequence` can index correctly
- `site_indices`: (5318,) array of 0-based `seq_idx` values for each mutation site, derived from ANARCI mappings in `abagym_sequences.csv`
- `chains`: (5318,) array of `'H'` or `'L'` indicating which chain carries each mutation

In [9]:
antibody_df  = load_abagym_antibody(DATA_DIR)
sequences_df = load_abagym_sequences(DATA_DIR)
sabdab_df    = load_sabdab(DATA_DIR)

# Arrays used by residue-level extraction functions
site_indices = get_all_mutation_site_indices(antibody_df, sequences_df)
chains       = antibody_df['chains'].values        # (5318,) 'H' or 'L'
dms_names    = antibody_df['DMS_name'].tolist()    # (5318,) dataset name per mutation

# Sequence tuples for pooled embedding
mutant_sequences  = list(zip(antibody_df['mutant_heavy_seq'], antibody_df['mutant_light_seq']))
sabdab_sequences  = list(zip(sabdab_df['heavy_seq'], sabdab_df['light_seq']))

# Wildtype sequences -- ordered by ABAGYM_DATASETS to fix the wt_index row mapping
wt_seq_by_name   = dict(zip(sequences_df['dms_name'],
                             zip(sequences_df['heavy_seq'], sequences_df['light_seq'])))
wildtype_sequences = [wt_seq_by_name[name] for name in ABAGYM_DATASETS]
wt_index           = {i: name for i, name in enumerate(ABAGYM_DATASETS)}

# Wildtype sequences expanded to 5318 rows (one per mutation row, for wtsite extraction)
wt_expanded_sequences = [wt_seq_by_name[name] for name in dms_names]

print(f"AbAgym mutations:      {len(antibody_df)}")
print(f"SAbDab entries:        {len(sabdab_df)}")
print(f"Wildtype antibodies:   {len(wildtype_sequences)}")
print(f"Site indices shape:    {site_indices.shape}")
print(f"Wt expanded length:    {len(wt_expanded_sequences)}")

AbAgym mutations:      5318
SAbDab entries:        491
Wildtype antibodies:   5
Site indices shape:    (5318,)
Wt expanded length:    5318


## ESM-2: Sequence-Level Embeddings (AbAgym)

Loads ESM-2 650M and generates mean-pooled embeddings for all 5318 mutant sequences and the 5 wildtype sequences. Heavy and light chains are embedded in separate forward passes and concatenated: concat(mean_pool(H), mean_pool(L)) = 2560-dim. This produces two progress bars per call -- one for heavy, one for light.

The wildtype index (`esm2_abagym_wildtype_index.json`) maps tensor row → antibody name and is required by `compute_delta_sequence` in the delta cell below.

Confirmed output:
- `esm2_abagym.pt`: (5318, 2560) -- H pass: 6:02 at 14.65 seq/s, L pass: 6:32 at 13.54 seq/s
- `esm2_abagym_wildtype.pt`: (5, 2560) -- near-instant (5 sequences)

In [10]:
esm_model, esm_alphabet, esm_batch_converter = esm2.load_esm2(DEVICE)
repr_layer = esm2.get_repr_layer(esm_model)
print(f"ESM-2 loaded. repr_layer={repr_layer}")

# --- Mutant AbAgym (5318, 2560) ---
mutant_path = EMBEDDING_DIR / 'esm2_abagym.pt'
if mutant_path.exists():
    t = torch.load(mutant_path, map_location='cpu')
    print(f"esm2_abagym.pt exists: {tuple(t.shape)}")
    assert t.shape == (5318, 2560)
else:
    print("Generating esm2_abagym.pt ...")
    t = esm2.embed_sequences_pooled(
        esm_model, esm_alphabet, esm_batch_converter,
        mutant_sequences, batch_size=32, device=DEVICE,
    )
    torch.save(t, mutant_path)
    print(f"Saved esm2_abagym.pt: {tuple(t.shape)}")

# --- Wildtype AbAgym (5, 2560) ---
wt_path = EMBEDDING_DIR / 'esm2_abagym_wildtype.pt'
if wt_path.exists():
    wt_t = torch.load(wt_path, map_location='cpu')
    print(f"esm2_abagym_wildtype.pt exists: {tuple(wt_t.shape)}")
    assert wt_t.shape == (5, 2560)
else:
    print("Generating esm2_abagym_wildtype.pt ...")
    wt_t = esm2.embed_sequences_pooled(
        esm_model, esm_alphabet, esm_batch_converter,
        wildtype_sequences, batch_size=5, device=DEVICE,
    )
    torch.save(wt_t, wt_path)
    save_index(wt_index, EMBEDDING_DIR / 'esm2_abagym_wildtype_index.json')
    print(f"Saved esm2_abagym_wildtype.pt: {tuple(wt_t.shape)}")

ESM-2 loaded. repr_layer=33
Generating esm2_abagym.pt ...


100%|██████████| 5318/5318 [06:32<00:00, 13.54seq/s]


Saved esm2_abagym.pt: (5318, 2560)
Generating esm2_abagym_wildtype.pt ...


100%|██████████| 5/5 [00:00<00:00, 12.83seq/s]

Saved esm2_abagym_wildtype.pt: (5, 2560)


## ESM-2: Sequence-Level Embeddings (SAbDab)

Same pooling strategy as AbAgym. 491 antibody sequences, 2560-dim output. Index maps row → `Antibody_ID` string for traceability.

Confirmed output:
- `esm2_sabdab.pt`: (491, 2560)

In [12]:
sabdab_path = EMBEDDING_DIR / 'esm2_sabdab.pt'
if sabdab_path.exists():
    t = torch.load(sabdab_path, map_location='cpu')
    print(f"esm2_sabdab.pt exists: {tuple(t.shape)}")
    assert t.shape == (491, 2560)
else:
    print("Generating esm2_sabdab.pt ...")
    t = esm2.embed_sequences_pooled(
        esm_model, esm_alphabet, esm_batch_converter,
        sabdab_sequences, batch_size=32, device=DEVICE,
    )
    torch.save(t, sabdab_path)
    sabdab_idx = {i: ab_id for i, ab_id in enumerate(sabdab_df['Antibody_ID'])}
    save_index(sabdab_idx, EMBEDDING_DIR / 'esm2_sabdab_index.json')
    print(f"Saved esm2_sabdab.pt: {tuple(t.shape)}")

esm2_sabdab.pt exists: (491, 2560)


## ESM-2: Residue-Level Embeddings (AbAgym)

Extracts the token embedding at each mutation site -- a single 1280-dim vector per sequence.

- **mutsite**: token at `seq_idx + 1` (BOS offset) from the **mutant** sequence -- represents the model's encoding of the mutant amino acid in context
- **wtsite**: same token position from the **wildtype** sequence -- represents the model's encoding of the original amino acid at that site

Row i of both tensors corresponds to the same mutation (same site, same antibody context). The residue-level delta (mutsite - wtsite) isolates what the model sees as changed between wildtype and mutant at the exact mutation position.

Note: unlike sequence-level pooling, residue extraction embeds only one chain per mutation (the chain carrying the mutation). Single progress bar per tensor.

Confirmed output:
- `esm2_abagym_residue_mutsite.pt`: (5318, 1280) -- 7:07 at 12.44 seq/s
- `esm2_abagym_residue_wtsite.pt`: (5318, 1280) -- 7:48 at 11.35 seq/s

In [13]:
mutsite_path = EMBEDDING_DIR / 'esm2_abagym_residue_mutsite.pt'
wtsite_path  = EMBEDDING_DIR / 'esm2_abagym_residue_wtsite.pt'

if mutsite_path.exists():
    t = torch.load(mutsite_path, map_location='cpu')
    print(f"esm2_abagym_residue_mutsite.pt exists: {tuple(t.shape)}")
    assert t.shape == (5318, 1280)
else:
    print("Generating esm2_abagym_residue_mutsite.pt ...")
    mutsite = esm2.embed_sequences_residue(
        esm_model, esm_alphabet, esm_batch_converter,
        mutant_sequences, site_indices, chains,
        batch_size=32, device=DEVICE,
    )
    torch.save(mutsite, mutsite_path)
    print(f"Saved esm2_abagym_residue_mutsite.pt: {tuple(mutsite.shape)}")

if wtsite_path.exists():
    t = torch.load(wtsite_path, map_location='cpu')
    print(f"esm2_abagym_residue_wtsite.pt exists: {tuple(t.shape)}")
    assert t.shape == (5318, 1280)
else:
    print("Generating esm2_abagym_residue_wtsite.pt ...")
    wtsite = esm2.embed_sequences_residue(
        esm_model, esm_alphabet, esm_batch_converter,
        wt_expanded_sequences, site_indices, chains,
        batch_size=32, device=DEVICE,
    )
    torch.save(wtsite, wtsite_path)
    print(f"Saved esm2_abagym_residue_wtsite.pt: {tuple(wtsite.shape)}")

Generating esm2_abagym_residue_mutsite.pt ...


100%|██████████| 5318/5318 [07:07<00:00, 12.44seq/s]


Saved esm2_abagym_residue_mutsite.pt: (5318, 1280)
Generating esm2_abagym_residue_wtsite.pt ...


100%|██████████| 5318/5318 [07:48<00:00, 11.35seq/s]

Saved esm2_abagym_residue_wtsite.pt: (5318, 1280)


## ESM-2: Delta Embeddings

Computes delta embeddings by subtraction. No forward passes -- operates on cached tensors only.

- **Sequence-level delta**: `mutant_emb[i] - wildtype_emb[wt_row]`, where `wt_row` is the row in the wildtype tensor for the antibody that mutation i belongs to. Shape (5318, 2560). Used in Experiment 4.
- **Residue-level delta**: `mutsite[i] - wtsite[i]`, element-wise. Shape (5318, 1280). Used in Experiments 2, 3, 5, 6.

Confirmed output:
- `esm2_abagym_delta.pt`: (5318, 2560)
- `esm2_abagym_residue_delta.pt`: (5318, 1280)

In [14]:
# --- Sequence-level delta (5318, 2560) ---
mutant_t = torch.load(EMBEDDING_DIR / 'esm2_abagym.pt',          map_location='cpu')
wt_t     = torch.load(EMBEDDING_DIR / 'esm2_abagym_wildtype.pt', map_location='cpu')
wt_idx   = load_index(EMBEDDING_DIR / 'esm2_abagym_wildtype_index.json')

delta_seq = compute_delta_sequence(mutant_t, wt_t, dms_names, wt_idx)
torch.save(delta_seq, EMBEDDING_DIR / 'esm2_abagym_delta.pt')
print(f"esm2_abagym_delta.pt: {tuple(delta_seq.shape)}")

# --- Residue-level delta (5318, 1280) ---
mutsite = torch.load(EMBEDDING_DIR / 'esm2_abagym_residue_mutsite.pt', map_location='cpu')
wtsite  = torch.load(EMBEDDING_DIR / 'esm2_abagym_residue_wtsite.pt',  map_location='cpu')
delta_res = compute_delta_residue(mutsite, wtsite)
torch.save(delta_res, EMBEDDING_DIR / 'esm2_abagym_residue_delta.pt')
print(f"esm2_abagym_residue_delta.pt: {tuple(delta_res.shape)}")

esm2_abagym_delta.pt: (5318, 2560)
esm2_abagym_residue_delta.pt: (5318, 1280)


## AbLang2: Sequence-Level Embeddings (AbAgym)

Loads AbLang2-paired and generates pooled embeddings for all mutant and wildtype sequences. Unlike ESM-2, both chains are processed in a single forward pass (`VH|VL` input). Chain masks separate heavy and light tokens; each is mean-pooled and concatenated: concat(mean_pool(H), mean_pool(L)) = 960-dim. Single progress bar per call (one joint pass, not two).

AbLang2 is ~3x faster than ESM-2 at equivalent batch size, consistent with the 15x parameter count difference (44M vs 651M).

Confirmed output:
- `ablang2_abagym.pt`: (5318, 960) -- 2:02 at 43.44 seq/s
- `ablang2_abagym_wildtype.pt`: (5, 960) -- near-instant

In [15]:
ablang_model = ablang2.load_ablang2(DEVICE)
print(f"AbLang2 loaded. Parameters: {sum(p.numel() for p in ablang_model.AbRep.parameters()):,}")

# --- Mutant AbAgym (5318, 960) ---
mutant_path = EMBEDDING_DIR / 'ablang2_abagym.pt'
if mutant_path.exists():
    t = torch.load(mutant_path, map_location='cpu')
    print(f"ablang2_abagym.pt exists: {tuple(t.shape)}")
    assert t.shape == (5318, 960)
else:
    print("Generating ablang2_abagym.pt ...")
    t = ablang2.embed_sequences_pooled(
        ablang_model, mutant_sequences, batch_size=32, device=DEVICE,
    )
    torch.save(t, mutant_path)
    print(f"Saved ablang2_abagym.pt: {tuple(t.shape)}")

# --- Wildtype AbAgym (5, 960) ---
wt_path = EMBEDDING_DIR / 'ablang2_abagym_wildtype.pt'
if wt_path.exists():
    wt_t = torch.load(wt_path, map_location='cpu')
    print(f"ablang2_abagym_wildtype.pt exists: {tuple(wt_t.shape)}")
    assert wt_t.shape == (5, 960)
else:
    print("Generating ablang2_abagym_wildtype.pt ...")
    wt_t = ablang2.embed_sequences_pooled(
        ablang_model, wildtype_sequences, batch_size=5, device=DEVICE,
    )
    torch.save(wt_t, wt_path)
    save_index(wt_index, EMBEDDING_DIR / 'ablang2_abagym_wildtype_index.json')
    print(f"Saved ablang2_abagym_wildtype.pt: {tuple(wt_t.shape)}")

AbLang2 loaded. Parameters: 44,348,304
Generating ablang2_abagym.pt ...


100%|██████████| 5318/5318 [02:02<00:00, 43.44seq/s]


Saved ablang2_abagym.pt: (5318, 960)
Generating ablang2_abagym_wildtype.pt ...


100%|██████████| 5/5 [00:00<00:00,  7.30seq/s]

Saved ablang2_abagym_wildtype.pt: (5, 960)


## AbLang2: Sequence-Level Embeddings (SAbDab)

Same as AbAgym but for 491 SAbDab antibody-antigen pairs. Output shape (491, 960).

Confirmed output:
- `ablang2_sabdab.pt`: (491, 960) -- 14 sec at 33.22 seq/s

In [16]:
sabdab_path = EMBEDDING_DIR / 'ablang2_sabdab.pt'
if sabdab_path.exists():
    t = torch.load(sabdab_path, map_location='cpu')
    print(f"ablang2_sabdab.pt exists: {tuple(t.shape)}")
    assert t.shape == (491, 960)
else:
    print("Generating ablang2_sabdab.pt ...")
    t = ablang2.embed_sequences_pooled(
        ablang_model, sabdab_sequences, batch_size=32, device=DEVICE,
    )
    torch.save(t, sabdab_path)
    sabdab_idx = {i: ab_id for i, ab_id in enumerate(sabdab_df['Antibody_ID'])}
    save_index(sabdab_idx, EMBEDDING_DIR / 'ablang2_sabdab_index.json')
    print(f"Saved ablang2_sabdab.pt: {tuple(t.shape)}")

Generating ablang2_sabdab.pt ...


100%|██████████| 491/491 [00:14<00:00, 33.22seq/s]

Saved ablang2_sabdab.pt: (491, 960)


## AbLang2: Residue-Level Embeddings (AbAgym)

Same extraction logic as ESM-2 but with different token position arithmetic. AbLang2 has no BOS token, so:
- Heavy chain mutation at `seq_idx`: `token_pos = seq_idx` (no offset)
- Light chain mutation at `seq_idx`: `token_pos = len(heavy_seq) + 1 + seq_idx` (+1 for the SEP token)

Both chains are tokenized together in a single pass (`VH|VL`), and the token at the mutation position is extracted from the joint representation. Unlike ESM-2 residue extraction (which embeds only the mutated chain), AbLang2 always embeds both chains together -- the joint representation is a feature, not overhead.

Confirmed output:
- `ablang2_abagym_residue_mutsite.pt`: (5318, 480) -- 2:04 at 42.75 seq/s
- `ablang2_abagym_residue_wtsite.pt`: (5318, 480) -- 2:21 at 37.66 seq/s

In [17]:
mutsite_path = EMBEDDING_DIR / 'ablang2_abagym_residue_mutsite.pt'
wtsite_path  = EMBEDDING_DIR / 'ablang2_abagym_residue_wtsite.pt'

if mutsite_path.exists():
    t = torch.load(mutsite_path, map_location='cpu')
    print(f"ablang2_abagym_residue_mutsite.pt exists: {tuple(t.shape)}")
    assert t.shape == (5318, 480)
else:
    print("Generating ablang2_abagym_residue_mutsite.pt ...")
    mutsite = ablang2.embed_sequences_residue(
        ablang_model, mutant_sequences, site_indices, chains,
        batch_size=32, device=DEVICE,
    )
    torch.save(mutsite, mutsite_path)
    print(f"Saved ablang2_abagym_residue_mutsite.pt: {tuple(mutsite.shape)}")

if wtsite_path.exists():
    t = torch.load(wtsite_path, map_location='cpu')
    print(f"ablang2_abagym_residue_wtsite.pt exists: {tuple(t.shape)}")
    assert t.shape == (5318, 480)
else:
    print("Generating ablang2_abagym_residue_wtsite.pt ...")
    wtsite = ablang2.embed_sequences_residue(
        ablang_model, wt_expanded_sequences, site_indices, chains,
        batch_size=32, device=DEVICE,
    )
    torch.save(wtsite, wtsite_path)
    print(f"Saved ablang2_abagym_residue_wtsite.pt: {tuple(wtsite.shape)}")

Generating ablang2_abagym_residue_mutsite.pt ...


100%|██████████| 5318/5318 [02:04<00:00, 42.75seq/s]


Saved ablang2_abagym_residue_mutsite.pt: (5318, 480)
Generating ablang2_abagym_residue_wtsite.pt ...


100%|██████████| 5318/5318 [02:21<00:00, 37.66seq/s]

Saved ablang2_abagym_residue_wtsite.pt: (5318, 480)


## AbLang2: Delta Embeddings

Same subtraction logic as ESM-2. No forward passes.

Confirmed output:
- `ablang2_abagym_delta.pt`: (5318, 960)
- `ablang2_abagym_residue_delta.pt`: (5318, 480)

In [18]:
# --- Sequence-level delta (5318, 960) ---
mutant_t = torch.load(EMBEDDING_DIR / 'ablang2_abagym.pt',          map_location='cpu')
wt_t     = torch.load(EMBEDDING_DIR / 'ablang2_abagym_wildtype.pt', map_location='cpu')
wt_idx   = load_index(EMBEDDING_DIR / 'ablang2_abagym_wildtype_index.json')

delta_seq = compute_delta_sequence(mutant_t, wt_t, dms_names, wt_idx)
torch.save(delta_seq, EMBEDDING_DIR / 'ablang2_abagym_delta.pt')
print(f"ablang2_abagym_delta.pt: {tuple(delta_seq.shape)}")

# --- Residue-level delta (5318, 480) ---
mutsite = torch.load(EMBEDDING_DIR / 'ablang2_abagym_residue_mutsite.pt', map_location='cpu')
wtsite  = torch.load(EMBEDDING_DIR / 'ablang2_abagym_residue_wtsite.pt',  map_location='cpu')
delta_res = compute_delta_residue(mutsite, wtsite)
torch.save(delta_res, EMBEDDING_DIR / 'ablang2_abagym_residue_delta.pt')
print(f"ablang2_abagym_residue_delta.pt: {tuple(delta_res.shape)}")

ablang2_abagym_delta.pt: (5318, 960)
ablang2_abagym_residue_delta.pt: (5318, 480)


## Verification

Checks all 14 output tensors for:
1. Correct shape
2. No NaN or Inf values
3. Nonzero delta at row 0 (Ang2_2017_G6 H:P100A) -- confirms mutant and wildtype embeddings are not identical

Confirmed: all 14 tensors passed. No NaN or Inf in any tensor.

Spot check -- row 0 (Ang2_2017_G6 H:P100A):
- ESM-2 sequence delta norm: 0.0720
- AbLang2 sequence delta norm: 0.0728

Both models produce similar-magnitude deltas for the same mutation, suggesting the two embedding spaces are comparably sensitive to this substitution at the sequence level. Whether this holds across the full distribution is the subject of NB04 EDA.

In [19]:
expected = {
    'esm2_abagym.pt':                     (5318, 2560),
    'esm2_abagym_wildtype.pt':            (5,    2560),
    'esm2_sabdab.pt':                     (491,  2560),
    'esm2_abagym_residue_mutsite.pt':     (5318, 1280),
    'esm2_abagym_residue_wtsite.pt':      (5318, 1280),
    'esm2_abagym_delta.pt':               (5318, 2560),
    'esm2_abagym_residue_delta.pt':       (5318, 1280),
    'ablang2_abagym.pt':                  (5318,  960),
    'ablang2_abagym_wildtype.pt':         (5,     960),
    'ablang2_sabdab.pt':                  (491,   960),
    'ablang2_abagym_residue_mutsite.pt':  (5318,  480),
    'ablang2_abagym_residue_wtsite.pt':   (5318,  480),
    'ablang2_abagym_delta.pt':            (5318,  960),
    'ablang2_abagym_residue_delta.pt':    (5318,  480),
}

all_ok = True
for fname, exp_shape in expected.items():
    path = EMBEDDING_DIR / fname
    if not path.exists():
        print(f"MISSING  {fname}")
        all_ok = False
        continue
    t = torch.load(path, map_location='cpu')
    nan_count  = torch.isnan(t).sum().item()
    inf_count  = torch.isinf(t).sum().item()
    shape_ok   = tuple(t.shape) == exp_shape
    clean      = nan_count == 0 and inf_count == 0
    status     = "OK  " if (shape_ok and clean) else "FAIL"
    print(f"{status}  {fname}: shape={tuple(t.shape)}  NaN={nan_count}  Inf={inf_count}")
    if not (shape_ok and clean):
        all_ok = False

# Spot check: delta norm at row 0 should be nonzero (mutant != wildtype)
esm_delta = torch.load(EMBEDDING_DIR / 'esm2_abagym_delta.pt', map_location='cpu')
ab_delta  = torch.load(EMBEDDING_DIR / 'ablang2_abagym_delta.pt', map_location='cpu')
print(f"\nSpot check -- row 0 (Ang2_2017_G6 H:P100A)")
print(f"  ESM-2 seq delta norm:    {esm_delta[0].norm().item():.4f}")
print(f"  AbLang2 seq delta norm:  {ab_delta[0].norm().item():.4f}")
assert esm_delta[0].norm().item() > 0
assert ab_delta[0].norm().item() > 0

print(f"\nAll checks passed: {all_ok}")

OK    esm2_abagym.pt: shape=(5318, 2560)  NaN=0  Inf=0
OK    esm2_abagym_wildtype.pt: shape=(5, 2560)  NaN=0  Inf=0
OK    esm2_sabdab.pt: shape=(491, 2560)  NaN=0  Inf=0
OK    esm2_abagym_residue_mutsite.pt: shape=(5318, 1280)  NaN=0  Inf=0
OK    esm2_abagym_residue_wtsite.pt: shape=(5318, 1280)  NaN=0  Inf=0
OK    esm2_abagym_delta.pt: shape=(5318, 2560)  NaN=0  Inf=0
OK    esm2_abagym_residue_delta.pt: shape=(5318, 1280)  NaN=0  Inf=0
OK    ablang2_abagym.pt: shape=(5318, 960)  NaN=0  Inf=0
OK    ablang2_abagym_wildtype.pt: shape=(5, 960)  NaN=0  Inf=0
OK    ablang2_sabdab.pt: shape=(491, 960)  NaN=0  Inf=0
OK    ablang2_abagym_residue_mutsite.pt: shape=(5318, 480)  NaN=0  Inf=0
OK    ablang2_abagym_residue_wtsite.pt: shape=(5318, 480)  NaN=0  Inf=0
OK    ablang2_abagym_delta.pt: shape=(5318, 960)  NaN=0  Inf=0
OK    ablang2_abagym_residue_delta.pt: shape=(5318, 480)  NaN=0  Inf=0

Spot check -- row 0 (Ang2_2017_G6 H:P100A)
  ESM-2 seq delta norm:    0.0720
  AbLang2 seq delta norm: 